In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings

from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
from meassures import get_meassure
from meassures import get_sensitive
from meassures import get_specificity
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.utils import resample

warnings.filterwarnings('ignore')

In [ ]:
# Se cargan los datos
data = pd.read_csv('full_dataset.csv')
data.head()

In [ ]:
# Sobre y sub muestreo

# Se separan los datos en conjuntos por clase
major_class = data[data.classes == 0]
minor_class = data[data.classes == 1]

# Se hace un sobremuestreo sobre la clase minoritaria
minor_class_upsampled = resample(minor_class, replace=True, n_samples=380, random_state=42)

# Se unen los conjuntos en un nuevo conjunto ?nico de clases
data_new = pd.concat([major_class, minor_class_upsampled])

# Se separan caracter?sticas de etiquetas
features = data_new.iloc[:, 0:-1]
target = data_new.iloc[:, -1]

# Se hace sub muestreo de conjunto mayoritario
rus = RandomUnderSampler(random_state=0)
features, target = rus.fit_resample(features, target)
print('Tama?o de los nuevos conjuntos de clases: {0}'.format(Counter(target)))

In [ ]:
pca = PCA(.93)
pca = pca.fit(features)
print("Explained Variance: {0}".format(pca.explained_variance_ratio_))
print(pca.components_)

# RandomForest Classifier

In [ ]:
for n_component in [1, 2, 5, 10]:
    pca = PCA(n_components=n_component)
    new_features = pca.fit_transform(features)
    fold = 6
    np.random.seed(12345)
    efficiency_train = np.zeros(fold)
    efficiency_validation = np.zeros(fold)

    skf = StratifiedKFold(n_splits=fold, random_state=None, shuffle=False)

    j = 0
    for train, test in skf.split(new_features, target):
        x_train = features[train, :]
        y_train = target[train]
        x_test = features[test, :]
        y_test = target[test]

        # Random Forest model
        model = RandomForestClassifier(n_estimators=20)
        model = model.fit(x_train, y_train)

        # Realizamos predicciones
        y_est_train = model.predict(x_train)
        y_est_test = model.predict(x_test)

        # Evaluamos las predicciones del modelo con los datos de test
        efficiency_train[j] = np.mean(y_est_train == y_train)
        efficiency_validation[j] = np.mean(y_est_test == y_test)
        
        # Medimos Sensibilidad y Especificidad
        tp, fp, tn, fn = get_meassure(y_test, y_est_test)
        sensitive = get_sensitive(tp, fn)
        specificity = get_specificity(tn, fp)
        
        j += 1
    
    print('-------------------------------------')
    print('N?mero de componentes: {0}'.format(n_component))
    print('-------------------------------------')
    print('-------------------------------------')
    print('Entrenamiento: {0} +- {1}'.format(str(np.mean(efficiency_train)), str(np.std(efficiency_train))))
    print('Validaci?n: {0} +- {1}'.format(str(np.mean(efficiency_validation)), str(np.std(efficiency_validation))))
    print('-------------------------------------')
    print('Sensibilidad: {0}, Especificidad: {1}'.format(sensitive, specificity))
    print('-------------------------------------')

# ANN

In [ ]:
for n_component in [1, 2, 5, 10]:
    pca = PCA(n_components=n_component)
    new_features = pca.fit_transform(features)
    fold = 3
    np.random.seed(12345)
    efficiency_train = np.zeros(fold)
    efficiency_validation = np.zeros(fold)

    skf = StratifiedKFold(n_splits=fold, random_state=None, shuffle=False)

    j = 0
    for train, test in skf.split(new_features, target):
        x_train = features[train, :]
        y_train = target[train]
        x_test = features[test, :]
        y_test = target[test]

        # ANN model
        model = MLPClassifier(hidden_layer_sizes=(20, 20), activation='logistic', max_iter=500)
        model.fit(x_train, y_train)

        # Realizamos predicciones
        y_est_train = model.predict(x_train)
        y_est_test = model.predict(x_test)

        # Evaluamos las predicciones del modelo con los datos de test
        efficiency_train[j] = np.mean(y_est_train == y_train)
        efficiency_validation[j] = np.mean(y_est_test == y_test)
        
        # Medimos Sensibilidad y Especificidad
        tp, fp, tn, fn = get_meassure(y_test, y_est_test)
        sensitive = get_sensitive(tp, fn)
        specificity = get_specificity(tn, fp)
        
        j += 1

    
    print('-------------------------------------')
    print('N?mero de componentes: {0}'.format(n_component))
    print('-------------------------------------')
    print('-------------------------------------')
    print('Entrenamiento: {0} +- {1}'.format(str(np.mean(efficiency_train)), str(np.std(efficiency_train))))
    print('Validaci?n: {0} +- {1}'.format(str(np.mean(efficiency_validation)), str(np.std(efficiency_validation))))
    print('-------------------------------------')
    print('Sensibilidad: {0}, Especificidad: {1}'.format(sensitive, specificity))
    print('-------------------------------------')

# SVM

In [ ]:
for n_component in [1, 2, 5, 10]:
    pca = PCA(n_components=n_component)
    new_features = pca.fit_transform(features)
    fold = 6
    np.random.seed(12345)
    efficiency_train = np.zeros(fold)
    efficiency_validation = np.zeros(fold)
    support_vectors_percent = np.zeros(fold)

    skf = StratifiedKFold(n_splits=fold, random_state=None, shuffle=False)

    j = 0
    for train, test in skf.split(new_features, target):
        x_train = features[train, :]
        y_train = target[train]
        x_test = features[test, :]
        y_test = target[test]

        # Random Forest model
        model = SVC(gamma=1, C=0.01, kernel='linear', decision_function_shape='ovo')
        model = model.fit(x_train, y_train)

        # Realizamos predicciones
        y_est_train = model.predict(x_train)
        y_est_test = model.predict(x_test)

        # Evaluamos las predicciones del modelo con los datos de test
        efficiency_train[j] = np.mean(y_est_train.ravel() == y_train.ravel())
        efficiency_validation[j] = np.mean(y_est_test.ravel() == y_test.ravel())
        support_vectors_percent[j] = len(model.support_vectors_) / len(x_train)
        
        # Medimos Sensibilidad y Especificidad
        tp, fp, tn, fn = get_meassure(y_test, y_est_test)
        sensitive = get_sensitive(tp, fn)
        specificity = get_specificity(tn, fp)
        
        j += 1
    
    print('-------------------------------------')
    print('N?mero de componentes: {0}'.format(n_component))
    print('-------------------------------------')
    print('-------------------------------------')
    print('Entrenamiento: {0} +- {1}'.format(str(np.mean(efficiency_train)), str(np.std(efficiency_train))))
    print('Validaci?n: {0} +- {1}'.format(str(np.mean(efficiency_validation)), str(np.std(efficiency_validation))))
    print('% de Vectores de Soporte = {0}'.format(np.mean(support_vectors_percent))) 
    print('-------------------------------------')
    print('Sensibilidad: {0}, Especificidad: {1}'.format(sensitive, specificity))
    print('-------------------------------------')